<a href="https://colab.research.google.com/github/Deepanshu-8126/Pandas/blob/main/Pandas_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 apply(), map(), applymap() / map() on DataFrame, Sorting

 # why this three important
 1- at every row caculation must
 2- tranform the value
 3- custom logic

 # map -  use in series
 work - transform evry element

 # apply - both series or dataframe
 work - custom function make

 # applymap - whole dataframe
 # at every cell make function


In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/Titanic-Dataset.csv')

# cleaning
df['Age'].fillna(df['Age'].median(), inplace=True)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
df.drop(columns=['Cabin'], inplace=True)

print(df.shape)
print(df.head())

(891, 11)
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Embarked  
0      0         A/5 21171   7.2500        S  
1      0          PC 17599  71.2833        C  
2      0  STON/O2. 3101282   7.9250        S  
3      0            113803  53.1000        S  
4      0            373450   8.0500        S  


/tmp/ipykernel_9611/3808118066.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)
/tmp/ipykernel_9611/3808118066.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', tr

In [4]:
# 5.1 - map() : at series element wise transform
# one value replace with other

# example - in titanic dataset male/female change them into 0 /1

gender_map = {'male':0,'female':1}
df['Sex_encoded']=df['Sex'].map(gender_map)

print(df[['Sex', 'Sex_encoded']].head(8))

      Sex  Sex_encoded
0    male            0
1  female            1
2  female            1
3  female            1
4    male            0
5    male            0
6    male            0
7    male            0


In [5]:
# way 2- embarked port_map
# S = Southampton, C = Cherbourg, Q = Queenstown
port_map = {'S': 'Southampton', 'C': 'Cherbourg', 'Q': 'Queenstown'}
df['Port_Name'] = df['Embarked'] .map(port_map)

print(df[['Embarked', 'Port_Name']].value_counts())

Embarked  Port_Name  
S         Southampton    646
C         Cherbourg      168
Q         Queenstown      77
Name: count, dtype: int64


In [6]:
# way 3- function to map()
# qu - basis with age generation define
def get_generation(age):
    if age < 18:
        return 'Minor'
    elif age < 35:
        return 'Young Adult'
    elif age < 55:
        return 'Middle Aged'
    else:
        return 'Senior'

df['Generation']=df['Age'].map(get_generation)
print(df[['Age','Generation']].head(5))


    Age   Generation
0  22.0  Young Adult
1  38.0  Middle Aged
2  26.0  Young Adult
3  35.0  Middle Aged
4  35.0  Middle Aged


In [7]:
# 4 way - map with lambda()
# a small anonymous function that written in one line
# Normal function:
#   def double(x): return x * 2
# Lambda:
#   lambda x: x * 2


# example - Fare convert to rupees
df['Fare_inr']= df['Fare'].map(lambda x : round(x*105,2))
print(df[['Fare', 'Fare_inr']].head(5))

      Fare  Fare_inr
0   7.2500    761.25
1  71.2833   7484.75
2   7.9250    832.12
3  53.1000   5575.50
4   8.0500    845.25


In [8]:
# map() and nan
test = pd.Series(['male', 'female', 'unknown'])
result = test.map({'male': 0, 'female': 1})

# fix
result = test.map({'male': 0, 'female': 1}).fillna(-1)
print(result)


0    0.0
1    1.0
2   -1.0
dtype: float64


In [9]:
# 5.2 apply()

df['Survival_Status'] = df['Survived'].apply(
    lambda x: 'Survived' if x == 1 else 'Died'
)
print(df[['Survived', 'Survival_Status']].head(8))

   Survived Survival_Status
0         0            Died
1         1        Survived
2         1        Survived
3         1        Survived
4         0            Died
5         0            Died
6         0            Died
7         0            Died


In [10]:
# complex logic

def categorize_fare(fare):
    if fare < 10:
        return 'Budget'
    elif fare < 30:
        return 'Economy'
    elif fare < 100:
        return 'Business'
    else:
        return 'Luxury'

df['Fare_Category'] = df['Fare'].apply(categorize_fare)
print(df['Fare_Category'].value_counts())

Fare_Category
Budget      336
Economy     315
Business    187
Luxury       53
Name: count, dtype: int64


In [11]:
# apply on dataframe
# risk score for every passenger
def risk_score(row):
    score = 0
    if row['Pclass'] == 3:
        score += 3
    elif row['Pclass'] == 2:
        score += 2
    else:
        score += 1
    if row['Sex'] == 'male':
        score += 2
    if row['Age'] > 40:
        score += 1
    return score

df['Risk_Score'] = df.apply(risk_score, axis=1)
print(df[['Name', 'Sex', 'Pclass', 'Age', 'Risk_Score']].head(10))

                                                Name     Sex  Pclass   Age  \
0                            Braund, Mr. Owen Harris    male       3  22.0   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female       1  38.0   
2                             Heikkinen, Miss. Laina  female       3  26.0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female       1  35.0   
4                           Allen, Mr. William Henry    male       3  35.0   
5                                   Moran, Mr. James    male       3  28.0   
6                            McCarthy, Mr. Timothy J    male       1  54.0   
7                     Palsson, Master. Gosta Leonard    male       3   2.0   
8  Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)  female       3  27.0   
9                Nasser, Mrs. Nicholas (Adele Achem)  female       2  14.0   

   Risk_Score  
0           5  
1           1  
2           3  
3           1  
4           5  
5           5  
6           4  
7           5

In [12]:
# apply on dataframe - column wise ( axis = 0)
# range ( max - min )
numeric_df = df[['Age', 'Fare', 'SibSp', 'Parch']]

ranges = numeric_df.apply(lambda col: col.max() - col.min(), axis=0)
print(ranges)

Age       79.5800
Fare     512.3292
SibSp      8.0000
Parch      6.0000
dtype: float64


In [13]:
# every numeric column normalize ( 0 to 1 range under )
# normalisation - values 0 - 1
# formula  : x - min  / max - min

def normalize(col):
    return (col - col.min()) / (col.max() - col.min())

normalized = numeric_df.apply(normalize, axis=0)
print(normalized.head())

        Age      Fare  SibSp  Parch
0  0.271174  0.014151  0.125    0.0
1  0.472229  0.139136  0.125    0.0
2  0.321438  0.015469  0.000    0.0
3  0.434531  0.103644  0.125    0.0
4  0.434531  0.015713  0.000    0.0


In [14]:
# apply with mulitple return values
def passenger_profile(row):
    title = row['Name'].split(',')[1].split('.')[0].strip()
    family = row['SibSp'] + row['Parch']
    alone = 'Yes' if family == 0 else 'No'
    return pd.Series([title, family, alone],
                     index=['Title', 'Family_Size', 'Is_Alone'])

profile = df.apply(passenger_profile, axis=1)
df = pd.concat([df, profile], axis=1)

print(df[['Name', 'Title', 'Family_Size', 'Is_Alone']].head(8))

                                                Name   Title  Family_Size  \
0                            Braund, Mr. Owen Harris      Mr            1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...     Mrs            1   
2                             Heikkinen, Miss. Laina    Miss            0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)     Mrs            1   
4                           Allen, Mr. William Henry      Mr            0   
5                                   Moran, Mr. James      Mr            0   
6                            McCarthy, Mr. Timothy J      Mr            0   
7                     Palsson, Master. Gosta Leonard  Master            4   

  Is_Alone  
0       No  
1       No  
2      Yes  
3       No  
4      Yes  
5      Yes  
6      Yes  
7       No  


In [17]:
# apply vs map diffrenece

# map - only series
df['Sex_num'] = df['Sex'].map({'male': 0, 'female': 1})


# apply - on series  handle also complex logic
df['Sex_num'] = df['Sex'].apply(lambda x: 0 if x == 'male' else 1)

# apply() on DataFrame — row/column wise operation
df['Risk'] = df.apply(lambda row: row['Age'] * row['Pclass'], axis=1)


In [18]:
# 5.3 topic applymap() / map on dataframe
# whole dataframe every cell function apply df.map use


numeric_df = df[['Age', 'Fare']].copy()
rounded = numeric_df.map(lambda x : round (x,1))
print(rounded.head())

    Age  Fare
0  22.0   7.2
1  38.0  71.3
2  26.0   7.9
3  35.0  53.1
4  35.0   8.1


In [21]:
# string dataframe all values uppercase
str_df = df[['Sex','Embarked']].copy()
upper_df = str_df.map(lambda x : x.upper()  if isinstance(x,str) else x)
print(upper_df.head())

      Sex Embarked
0    MALE        S
1  FEMALE        C
2  FEMALE        S
3  FEMALE        S
4    MALE        S


In [25]:
# 5.4 sort_values() -- data sort
# ques - age basis ascending sort
# ascending - false then descending
df_sort = df.sort_values('Age',ascending = False)
print(df_sort[['Name','Age']].head(8))

                                     Name   Age
630  Barkworth, Mr. Algernon Henry Wilson  80.0
851                   Svensson, Mr. Johan  74.0
96              Goldschmidt, Mr. George B  71.0
493               Artagaveytia, Mr. Ramon  71.0
116                  Connors, Mr. Patrick  70.5
745          Crosby, Capt. Edward Gifford  70.0
672           Mitchell, Mr. Henry Michael  70.0
33                  Wheadon, Mr. Edward H  66.0


In [27]:
# multiple coulumns sort
df_sorted = df.sort_values(
    by = ['Pclass', 'Age'], ascending = [True , False]
)
print(df_sorted[['Name', 'Pclass', 'Age', 'Survived']].head(10))

                                     Name  Pclass   Age  Survived
630  Barkworth, Mr. Algernon Henry Wilson       1  80.0         1
96              Goldschmidt, Mr. George B       1  71.0         0
493               Artagaveytia, Mr. Ramon       1  71.0         0
745          Crosby, Capt. Edward Gifford       1  70.0         0
54         Ostby, Mr. Engelhart Cornelius       1  65.0         0
456             Millet, Mr. Francis Davis       1  65.0         0
438                     Fortune, Mr. Mark       1  64.0         0
545          Nicholson, Mr. Arthur Ernest       1  64.0         0
275     Andrews, Miss. Kornelia Theodosia       1  63.0         1
252             Stead, Mr. William Thomas       1  62.0         0


In [ ]:
#